# BT4012 Fraud Analytics — E0906431

## Final V22: learned edge-gated transaction ranking

This notebook is the reproducible entry point for `submission_edgegate_v22_experimental.csv`, the best identified **public** submission (ROC-AUC **0.95716**). The final team private leaderboard AUC is **0.94688**, but the corresponding submitted filename has not been independently verified; that score is not attributed to this file here.

The complete training implementations are in `final_model/`. Only the four original competition CSVs are required. No earlier submissions, local experiment caches, pretrained models or external labels are required. Python 3.12 and exact package versions are listed in `requirements.txt`.

The model combines tree, tabular neural and graph neural predictions, followed by learned graph agreement gates. Unknown labels never become negative training targets. All fitted transformations and model inputs respect the fitting cutoff. Full same-snapshot topology is a modelling assumption, not a real-time deployment claim.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display
from final_model.pipeline import load_data, train_predict, generate_submission, sha

ROOT = Path.cwd()
DATA = ROOT / "data/raw"
OUTPUT = ROOT / "outputs/v22_reproduction"
RUN_VALIDATION = True
OUTPUT.mkdir(parents=True, exist_ok=True)
context = load_data(DATA)
print("Original input hashes, identifiers, schema and snapshot alignment verified.")

Original input hashes, identifiers, schema and snapshot alignment verified.


## Data preprocessing and graph construction

Feature rows remain in their supplied order. Missing-label training rows retain their features but never contribute a supervised target. Graph-only nodes are appended in sorted identifier order; their time is inferred only from a component with one observed time anchor. Incoming adjacency uses destination rows and source columns. Identifiers and time are not model predictors.

Imputation and scaling are fitted only on past rows. One graph model may use unlabelled feature rows from past snapshots; the availability-matched graph model uses known target rows for its fitting statistics. Neither fits a transformation to future test features.

In [2]:
train, test = context["train"], context["test"]
summary = pd.Series({
    "training_rows": len(train), "known_labels": int(train.label.notna().sum()),
    "normal": int((train.label == 0).sum()), "illicit": int((train.label == 1).sum()),
    "unknown_labels": int(train.label.isna().sum()), "test_rows": len(test),
    "graph_nodes": len(context["node_ids"]), "directed_edges": len(context["source"]),
    "training_first_step": int(train.time_step.min()), "training_last_step": int(train.time_step.max()),
    "test_first_step": int(test.time_step.min()), "test_last_step": int(test.time_step.max()),
})
display(summary)

training_rows          141772
known_labels            31235
normal                  27591
illicit                  3644
unknown_labels         110537
test_rows               15329
graph_nodes            203769
directed_edges         234355
training_first_step         1
training_last_step         35
test_first_step            36
test_last_step             49
dtype: int64

## Fixed final model recipe

1. **Tree and full-snapshot graph ensemble:** XGBoost400 trees; raw and time-proxy-filtered CatBoost336/244 trees with seeds42/52/62; bidirectional GraphSAGE40 epochs with the same three seeds. The tree average and graph ensemble form V5.
2. **Tabular rank ensemble:** an eight-head shared MLP, width128, quantile preprocessing and24 epochs, seeds42/52/62. Combine75% rank(V5) with25% rank(MLP), giving V9.
3. **Availability-matched graph model:** three40-epoch quantile GraphSAGE models. V14 is90% V9 plus10% rank of their mean prediction.
4. **Recent supervised component:** three800-tree XGBoost models with half-life10 weighting and past-only removal of strong time proxies. V21 combines95% V14 and5% rank of mean margins, then applies a10% incoming-neighbor logit correction.
5. **Learned agreement gates:** six350-tree depth3 XGBoost models, three on direct edges and three on shared-parent pairs. Each uses102 symmetric endpoint features from32 past-selected quantile features and graph degrees. Direct training balances four actual endpoint-label cells; sibling training balances normal-normal, illicit-illicit and mixed pairs. Average probabilities before applying `max(2*p-1, 0)`.
6. Three anchored direct iterations at strength0.6, then one sibling correction at0.2 produce V22. Unknown intermediaries need no invented attributes or labels. The output is a ranking score, not a calibrated fraud probability.

All lengths and weights are fixed; running this notebook does not perform a new model search.

## Temporal validation

Fit through step30 and evaluate known labels at steps31–35. This is a **retrospective evaluation of the fixed final recipe**, not the originally adaptive upstream validation procedure. These periods were already used during experimentation, so the result is not an untouched holdout or a public-score forecast. No early stopping or preprocessing uses these query labels.

The first run trains the full chain for this cutoff. Re-execution reuses only checkpoints created by this reproduction pipeline in its output directory. Set `RUN_VALIDATION=False` only when generating the final submission without repeating this diagnostic.

In [3]:
if RUN_VALIDATION:
    validation_ids = np.flatnonzero(context["known"] & (context["times"] > 30) & (context["times"] <= 35))
    validation_prediction, validation_summary = train_predict(context, 30, validation_ids, OUTPUT)
    display(pd.Series(validation_summary).drop("raw_hashes"))
else:
    print("Historical diagnostic explicitly skipped; final fitting remains unchanged.")

Completed E04
Completed CAT_raw_s42
Completed CAT_raw_s52
Completed CAT_raw_s62
Completed CAT_drop_s42
Completed CAT_drop_s52
Completed CAT_drop_s62
Completed SAGE_s42
Completed SAGE_s52
Completed SAGE_s62
Completed C37_s42
Completed C37_s52
Completed C37_s62
TARGET_QNT_c30_s42: 24.5s, labels=26905, visible fit rows=26905
TARGET_QNT_c30_s52: 23.2s, labels=26905, visible fit rows=26905
TARGET_QNT_c30_s62: 24.3s, labels=26905, visible fit rows=26905
{
  "cutoff": 30,
  "known_fit_rows": 26905,
  "query_rows": 4330,
  "seeds": [
    42,
    52,
    62
  ],
  "seconds": 360.1958620420046,
  "raw_hashes": {
    "train.csv": "6f7e56883f0aa27147262a0d71954ad64702ca57f16fbf9ed58fae21a66dd706",
    "test.csv": "344e47c8cd8815f4072d7f6ecbecee8919ab648294a8eb4ed032a026c3eb91db",
    "txs_edgelist.csv": "a35053ba68a98e4382cae2ba65b9d9e36b23b6439e02dff084971b1b72a5156e",
    "sample_submission.csv": "3dba74ad6151053aa21f366fb9faf6da7393969a0ba6f8a2ebcec561871f739e"
  },
  "new_model_fits_this_invoc

cutoff                                                                           30
known_fit_rows                                                                26905
query_rows                                                                     4330
seeds                                                                  [42, 52, 62]
seconds                                                                  360.195862
new_model_fits_this_invocation                                                   25
validation                        Retrospective fixed final recipe on reused per...
auc                                                                        0.998651
dtype: object

## Final fitting and CSV generation

Refit the exact recipe on all observed labels through step35 and predict steps36–49. The four raw files are the only external inputs. The original archived CSV is not read. A fresh run trains25 models, including every upstream component; CPU execution uses at most two numerical threads. Runtime depends on hardware.

The final file uses the literal test `index` key and exact sample-template order, with unique IDs and finite continuous scores in[0,1]. Nothing is uploaded automatically.

In [4]:
submission_path, final_summary = generate_submission(context, OUTPUT)
display(pd.Series(final_summary).drop("raw_hashes"))
print("Generated:", submission_path.relative_to(ROOT))
display(pd.Series(json.loads((OUTPUT / "submission_checks.json").read_text())))

{
  "cutoff": 35,
  "known_fit_rows": 31235,
  "query_rows": 15329,
  "seeds": [
    42,
    52,
    62
  ],
  "seconds": 3.86209570801293,
  "raw_hashes": {
    "train.csv": "6f7e56883f0aa27147262a0d71954ad64702ca57f16fbf9ed58fae21a66dd706",
    "test.csv": "344e47c8cd8815f4072d7f6ecbecee8919ab648294a8eb4ed032a026c3eb91db",
    "txs_edgelist.csv": "a35053ba68a98e4382cae2ba65b9d9e36b23b6439e02dff084971b1b72a5156e",
    "sample_submission.csv": "3dba74ad6151053aa21f366fb9faf6da7393969a0ba6f8a2ebcec561871f739e"
  },
  "new_model_fits_this_invocation": 0,
  "validation": "Retrospective fixed final recipe on reused periods; not original adaptive metrics or an untouched holdout."
}


cutoff                                                                           35
known_fit_rows                                                                31235
query_rows                                                                    15329
seeds                                                                  [42, 52, 62]
seconds                                                                    3.862096
new_model_fits_this_invocation                                                    0
validation                        Retrospective fixed final recipe on reused per...
dtype: object

Generated: outputs/v22_reproduction/submission_edgegate_v22_reproduced.csv


passed                                                               True
rows                                                                15329
columns                                                   [index, target]
sha256                  feaff00f7cf88b3be6e4b887d5a8b01b90b7eb3ab52940...
minimum                                                          0.007838
maximum                                                          0.998421
template_order_exact                                                 True
csv_roundtrip_exact                                                  True
dtype: object

## Results, limitations and reproducibility

The competition's verified public leaderboard best is0.95716 for V22; later V23/V24 did not improve it. The modelling process reused temporal windows adaptively, and strong local scores did not consistently transfer to the leaderboard. Report these limitations explicitly.

`final_model/reproduction_report.json` records the clean reproduction check against the archived V22 output and the tested environment. Floating-point libraries and hardware can produce small differences in neural training; the archived submission hash identifies the original submitted artifact. Raw data, fitted models and prediction CSVs are intentionally excluded from Git. The generated local CSV is the output to inspect; this notebook does not contact Kaggle.

The original baseline notebook `bt4012.ipynb` remains for comparison. The report should cite this repository and describe the final approach and its chronological validation without presenting reused diagnostics as independent test performance.